In [1]:
txt = ""
with open("./data/processed_documents/LEGITEXT000006070721.md",'r',encoding="utf-8") as d:
    txt = d.read()

In [110]:
import re
from uuid import uuid4
from langchain_chroma import Chroma
from langchain_core.documents import Document
import numpy as np

In [3]:
chunks = re.split(r"## Article",txt)

In [22]:
chunks[785].split("\n",1)

[' 512',
 "\nPour les majeurs protégés, les comptes de gestion sont vérifiés et approuvés annuellement par le subrogé tuteur lorsqu'il en a été nommé un ou par le conseil de famille lorsqu'il est fait application de l'article 457. Lorsque plusieurs personnes ont été désignées dans les conditions de l'article 447 pour la gestion patrimoniale, les comptes annuels de gestion doivent être signés par chacune d'elles, ce qui vaut approbation. En cas de difficulté, le juge statue sur la conformité des comptes à la requête de l'une des personnes chargées de la mesure de protection.\n\nPar dérogation au premier alinéa du présent article, lorsque l'importance et la composition du patrimoine de la personne protégée le justifient, le juge désigne, dès réception de l'inventaire du budget prévisionnel, un professionnel qualifié chargé de la vérification et de l'approbation des comptes dans des conditions fixées par décret en Conseil d'Etat. Le juge fixe dans sa décision les modalités selon lesquelle

In [33]:
splitted = map(lambda c: c.split("\n",1),chunks)

In [34]:
dnf = [(a[0],a[1]) for a in splitted]

In [35]:
dnf[:10]

[('## Code civil',
  "\n## Titre préliminaire : De la publication, des effets et de l'application des lois en général\n\n"),
 (' 1',
  "\nLes lois et, lorsqu'ils sont publiés au Journal officiel de la République française, les actes administratifs entrent en vigueur à la date qu'ils fixent ou, à défaut, le lendemain de leur publication. Toutefois, l'entrée en vigueur de celles de leurs dispositions dont l'exécution nécessite des mesures d'application est reportée à la date d'entrée en vigueur de ces mesures.\n\nEn cas d'urgence, entrent en vigueur dès leur publication les lois dont le décret de promulgation le prescrit et les actes administratifs pour lesquels le Gouvernement l'ordonne par une disposition spéciale.\n\nLes dispositions du présent article ne sont pas applicables aux actes individuels.\n\n"),
 (' 2',
  "\nLa loi ne dispose que pour l'avenir ; elle n'a point d'effet rétroactif.\n\n"),
 (' 3',
  "\nLes lois de police et de sûreté obligent tous ceux qui habitent le territoir

## vectorization

In [5]:
max([len(c) for c in chunks])

5469

we will use local embedding model : MODEL_ID = "nvidia/Nemotron-3-Embed-8B-BF16" for vectorization,
he scored very high on the RTEB multilingual and particulary on FRENCH legal retrieval which is very intresting for this 
project.

In [102]:
class CustomEmbedOpenRouter:

    endpoint = "https://openrouter.ai/api/v1/embeddings"
    model = "openai/text-embedding-3-small"

    def __init__(self,apikey=None):

        if apikey:
            self.apikey = apikey
            
    



    def embed_batch(self,chunks):
        response = requests.post("https://openrouter.ai/api/v1/embeddings",headers={
    "Authorization": f"Bearer {self.apikey}",
    "Content-Type": "application/json",
      },
      json={
        "model": "openai/text-embedding-3-small",
        "input": chunks
      }
                                )
    
        data = response.json()
        return data["data"]



    def embed_query(self,query:str):
        response = requests.post(
          "https://openrouter.ai/api/v1/embeddings",
          headers={
            "Authorization": f"Bearer {self.apikey}",
            "Content-Type": "application/json",
          },
          json={
            "model": "openai/text-embedding-3-small",
            "input": query
          }
        )
        
        data = response.json()
        embedding = data["data"][0]["embedding"]
        return embedding
                

    
    def embed_documents(self,chunks:str,batch_size=16):

        num_iter = len(chunks) // batch_size
        remainder = len(chunks) % batch_size

        res = []


        for i in range(num_iter):

            curr_batch = chunks[batch_size*i:batch_size*(i+1)]

            embeddings = self.embed_batch(curr_batch)
            for emb in embeddings:
                
                res.append(emb["embedding"])

        embeddings = self.embed_batch(chunks[-remainder:])
        for emb in embeddings:
                res.append(emb["embedding"])
        
        return res

In [103]:
store = Chroma(
    collection_name="articles_code_civil",
    embedding_function=CustomEmbed(),
    host="localhost",
)

In [36]:
docs = [Document(page_content=c[1],metadata={"source": f"Article {c[0]} du code civil"}) for c in dnf]

In [107]:
store.add_documents(documents=docs[19:])

['1681bfe1-397a-46b0-b126-d1f7eaac705e',
 '2bc492de-a49d-4c96-9269-7ae5b27d8567',
 'ce340b82-d54d-40ba-9a54-6b51e9736790',
 'eaf5e55d-9a0c-4e83-bc0b-ab30ce43da48',
 '7b41e747-959e-4280-a99c-9d8231b66349',
 '855a6116-7235-466d-8304-fb54b15db7a4',
 'a89bd5e7-3eb8-4a1a-97df-325e5c8af2ca',
 'fea411a1-15b0-4a7d-897d-5e810e3572db',
 '64bbd01c-d166-43d2-84e7-cfabbd2a56aa',
 '358f9249-2f1e-4a05-be1f-0f9dbde20309',
 'd7fc3b70-7eeb-4663-9f27-d62fc6e0e539',
 'a2a7aa1c-ce37-4997-a4af-73962a8a022c',
 '5c015d91-3f21-4e93-8cde-1972718cff69',
 '29ccf027-6794-4a44-a7a9-b7e058cadeee',
 '9c8331c3-4e9f-4a10-81ed-e2eb0bba7d95',
 'a884c434-12a2-4408-908b-d3d29fe55075',
 '9cca1c5c-06dc-4e5f-bc19-1acc7ea92592',
 '4f0987ae-5cdf-457f-94b7-63395c169e8e',
 'c2adc139-6776-4f63-bdc1-2f9c9545ac4e',
 '1a9a46ac-868f-4175-a09a-f5acb5a084e1',
 '9cc9503f-626b-4884-993c-1bf0aaa412f4',
 '63df10a1-b17e-4902-801e-c3550dfad86f',
 '6e41dbea-cf74-4347-bb0b-4c920073b5b9',
 '6cb06932-263e-4f95-aadc-776fddc16158',
 'bc9c7b26-0e83-

In [108]:
results = store.similarity_search(
    "Qu'est-ce qui est retroactif ?",
    k=2,
)

In [109]:
results

[Document(id='4fe91ac4-1292-46d3-984d-b50bea3325c5', metadata={'source': 'Article  1844-15-2 du code civil'}, page_content="\nLorsque la rétroactivité de la nullité d'une décision sociale est de nature à produire des effets manifestement excessifs pour l'intérêt social, les effets de cette nullité peuvent être différés.\n\n"),
 Document(id='b704e37e-db47-44c6-9da8-7d0fe1bd31bd', metadata={'source': 'Article  1304-7 du code civil'}, page_content="\nL'accomplissement de la condition résolutoire éteint rétroactivement l'obligation, sans remettre en cause, le cas échéant, les actes conservatoires et d'administration.\n\nLa rétroactivité n'a pas lieu si telle est la convention des parties ou si les prestations échangées ont trouvé leur utilité au fur et à mesure de l'exécution réciproque du contrat.\n\n## Section 2 : L'obligation à terme\n\n")]